# Objetivos

* Determinar la composición (frecuencia absoluta y porcentual) de las respuestas por pregunta.
* Determinar el número de preguntas basales según codificación.
  > Para ello se considera que el código de las preguntas, según el libro de código cuentan con la siguiente estructura <3 letras><3 dígitos ordinales> y si son no basales, letras ordinales tras los dígitos.
  > Es de notar que éstas cantidades también fueron comprobadas de forma manual para al menos 4 condiciones dentro de los módulos C y E.
* Determinar la cantidad de datos perdidos por pregunta.

Nota: Dentro del código se le llama "condition" a cada padecimiento o enfermedad, es posible que este término sea inexacto.

In [2]:
import pandas as pd 
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

In [3]:
df_sintomas_disc = pd.read_excel(r"C:\Users\Lan_1\Documents\VictorL_TesinaMACI\VictorL_TesinaMACI\Python\data\Sintomas_disc_Infante_Juvenil_n=1558.xlsx")

In [4]:
def cond_df(df, code, disp=False):
    code_cols = [i for i in df if i.startswith(code) and i[3].isnumeric()]
    root_cols = [True if len(i) <= 6 else False for i in code_cols]

    new_df = df[code_cols]
    df_root = pd.DataFrame({"question": code_cols, "root": root_cols})

    if disp:
        nan_display(new_df)

    return new_df, df_root

def nan_display(df):
        pd.set_option('display.max_rows', None)
        pd.set_option('display.max_columns', None)
        display(df.isnull().sum())

def comp_cols(df):
    rows = []
    for col in df.columns:
        counts = df[col].value_counts(dropna=False)
        for val, cnt in counts.items():
            rows.append([col, val, cnt, cnt/len(df)*100])
    return pd.DataFrame(rows, columns=["question", "value", "count", "pct"])

def comp_cols_pct(df):
    possible_values = sorted(df.stack().unique())
    rows = []
    for col in df.columns:
        counts = df[col].value_counts(dropna=False)
        row = [col] + [counts.get(val, 0)/len(df)*100 for val in possible_values]
        rows.append(row)
    return pd.DataFrame(rows, columns=["question"] + [f"pct_{val}" for val in possible_values])

def plot_comp(df_comp, condition):
    map_labels = {
        2: "2 - Sí",
        0: "0 - No",
        3: "3 - A veces / Algo",
        7: "7 - Se niega a responder",
        77: "77 - Se niega a responder",
        8: "8 - No aplicable",
        88: "88 - No aplicable",
        9: "9 - No sabe",
        99: "99 - No sabe"
    }
    
    map_colors = {
        "2 - Sí": "#2ca02c",
        "0 - No": "#d62728",
        "3 - A veces / Algo": "#1f77b4",
        "7 - Se niega a responder": "#9467bd",
        "77 - Se niega a responder": "#9467bd",
        "8 - No aplicable": "#8c564b",
        "88 - No aplicable": "#8c564b",
        "9 - No sabe": "#e377c2",
        "99 - No sabe": "#e377c2",
        "NaN": "#7f7f7f"
    }


    questions = df_comp["question"].unique()
    fig = go.Figure()

    for q in questions:
        sub = df_comp[df_comp["question"] == q].copy()
        sub["label"] = sub["value"].apply(
            lambda v: map_labels[v] if v in map_labels else ("NaN" if pd.isna(v) else str(v))
        )
        sub["color"] = sub["label"].apply(lambda l: map_colors.get(l, "#7f7f7f"))

        fig.add_trace(
            go.Pie(
                labels=sub["label"],
                values=sub["count"],
                hole=0.5,
                visible=False,
                name=q,
                marker=dict(colors=sub["color"])
            )
        )

    fig.data[0].visible = True

    buttons = []
    for i, q in enumerate(questions):
        vis = [False]*len(questions)
        vis[i] = True
        buttons.append(
            dict(
                label=q,
                method="update",
                args=[{"visible": vis}]
            )
        )

    fig.update_layout(
        updatemenus=[{"buttons": buttons}],
        showlegend=True,
        title='Composición de Preguntas - ' + condition
    )

    return fig

def report(dic):
    pd.set_option('display.max_rows', None)
    for code, info in dic.items():

        print(info["nombre"], "-", code)
        print('')
        print('COMPOSICIÓN DE RESPUESTAS')
        
        display(info['df_comp'])
        display(info['comp_plot'])
        print('')
        print('NÚMERO DE RESPUESTAS BASALES Y NO BASALES')
        
        print('Preguntas Basales: ', info['root_question_count'])
        print('Preguntas Totales: ', info['question_count'])
        print('Porcentaje de Preguntas Basales: ', info['root_question_percent'], '%')
        print('')
        print('NÚMERO DE RESPUESTAS PERDIDAS')
        
        display(info['df_comp'][info['df_comp']['value'].isna()])

        print('-'*500)

In [ ]:
full_dict = {
    "pag": {"nombre": "Agorafobia", "modulo": "A"},
    "psp": {"nombre": "Fobia específica", "modulo": "A"},
    "pso": {"nombre": "Fobia social", "modulo": "A"},
    "pga": {"nombre": "Ansiedad generalizada", "modulo": "A"},
    "psm": {"nombre": "Mutismo selectivo", "modulo": "A"},
    "ppa": {"nombre": "Pánico", "modulo": "A"},
    "ppt": {"nombre": "Trastorno por estrés postraumático", "modulo": "A"},
    "psa": {"nombre": "Ansiedad por separación", "modulo": "A"},
    "poc": {"nombre": "Trastorno obsesivo-compulsivo", "modulo": "A"},
    "pea": {"nombre": "Bulimia", "modulo": "B"},
    "pel": {"nombre": "Trastorno de eliminación", "modulo": "B"},
    "ppi": {"nombre": "Pica", "modulo": "B"},
    "ptc": {"nombre": "Trastorno de tics", "modulo": "B"},
    "ptr": {"nombre": "Tricotilomanía", "modulo": "B"},
    "pmd": {"nombre": "Depresión mayor o distimia", "modulo": "C"},
    "pma": {"nombre": "Manía o hipomanía", "modulo": "C"},
    "psz": {"nombre": "Esquizofrenia", "modulo": "D"},
    "pad": {"nombre": "Trastorno por déficit de atención e hiperactividad", "modulo": "E"},
    "pcd": {"nombre": "Trastorno de conducta", "modulo": "E"},
    "pod": {"nombre": "Trastorno de oposición desafiante", "modulo": "E"},
    "pal": {"nombre": "Abuso de alcohol", "modulo": "F"},
    "pmj": {"nombre": "Consumo de marihuana", "modulo": "F"},
    "psu": {"nombre": "Consumo de otras sustancias", "modulo": "F"},
    "pni": {"nombre": "Consumo de tabaco", "modulo": "F"}
}

result = {}

for code, info in full_dict.items():
    print(info["nombre"], "-", code)
    df_filtered,df_root = cond_df(df_sintomas_disc, code, disp=False)
    info["df"] = df_filtered
    info['df_root'] = df_root
    df_comp = comp_cols(df_filtered)
    info['df_comp'] = df_comp
    info['comp_plot'] = plot_comp(df_comp, info["nombre"] + " - " + code)
    trunc = [i[:6] for i in df_filtered.columns]
    uniques = list(dict.fromkeys(trunc))
    info['root_question_count'] = len(uniques)
    info['question_count'] = len(df_root)
    info['root_question_percent'] = len(uniques)/len(df_root)*100
    print("Done")
    print("-"*50)




Agorafobia - pag


In [ ]:
mod_C_dict = {k: v for k, v in full_dict.items() if v["modulo"] == "C"}
mod_E_dict = {k: v for k, v in full_dict.items() if v["modulo"] == "E"}

# Exploración para el Módulo C

In [ ]:
report(mod_C_dict)

# Exploración para el Módulo E

In [ ]:
report(mod_E_dict)

# ANEXO: EXPLORACIÓN COMPLETA

In [ ]:
report(full_dict)